# 노트북 8. Structured Output

> Phase 3 — 실전 기법: 챗봇을 똑똑하게 만드는 기술

LLM 출력을 사람이 읽는 게 아니라 코드가 파싱해야 한다면, 구조화된 출력이 필수입니다.
"JSON으로 답해줘"라고 프롬프트에 쓰는 것은 불안정합니다. 이 노트북에서는 확실한 방법을 배웁니다.

**학습 목표**
- 프롬프트 기반 JSON 요청의 불안정성을 이해한다
- google-genai의 response_mime_type과 response_schema를 사용할 수 있다
- LangChain의 with_structured_output()과 Pydantic 모델을 활용할 수 있다
- Pydantic 모델을 설계하여 복잡한 구조화 출력을 정의할 수 있다
- 파싱 실패 시 디버깅과 재시도 패턴을 적용할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | JSON 강제 + Pydantic 설계 + 실패 핸들링 | 읽고 실행 |
| Part 2 — 실습 | 스키마 정의 + 구조화 출력 + 스트리밍 | 직접 작성 |
| Part 3 — 챌린지 | 복잡한 중첩 스키마 설계 + 파싱 실패 대응 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai pydantic

import os
import json
from google import genai

# API 키 설정 (Colab 환경 기준)
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

print("환경 설정 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.9 MB/s eta 0:00:00
환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 왜 구조화된 출력이 필요한가?

LLM의 출력을 코드에서 처리해야 하는 경우가 많습니다.
- API 서버에서 JSON 응답을 반환할 때
- 분류 결과를 데이터베이스에 저장할 때
- 추출된 정보를 다른 시스템에 전달할 때

"JSON으로 답해줘"라는 프롬프트는 **불안정**합니다.
아래 코드는 프롬프트만으로 JSON을 요청했을 때의 문제를 보여줍니다.

In [ ]:
# 프롬프트만으로 JSON 요청 — 불안정한 방법
prompt = """다음 영화에 대한 정보를 JSON 형식으로 알려줘.
영화: 인셉션
필드: title, director, year, genre, rating(10점 만점)"""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"thinking_config": {"thinking_budget": 0}},
)

print("=== 모델 응답 ===")
print(response.text)

# JSON 파싱 시도
print("\n=== json.loads() 시도 ===")
try:
    data = json.loads(response.text)
    print(f"성공: {data}")
except json.JSONDecodeError as e:
    print(f"실패: {e}")
    print("(마크다운 코드블록, 설명 텍스트 등이 섞여 파싱 불가)")

=== 모델 응답 ===
```json
{
  "title": "인셉션",
  "director": "크리스토퍼 놀란",
  "year": 2010,
  "genre": ["SF", "액션", "스릴러"],
  "rating": 8.8
}
```

=== json.loads() 시도 ===
실패: Expecting value: line 1 column 1 (char 0)
(마크다운 코드블록, 설명 텍스트 등이 섞여 파싱 불가)


모델이 JSON을 반환하더라도 다음과 같은 문제가 발생할 수 있습니다:

| 문제 | 예시 |
|------|------|
| 마크다운 코드블록 | ` ```json\n{...}\n``` ` |
| 자연어 혼합 | `다음은 결과입니다: {"title": ...}` |
| 키 이름 불일치 | `movie_title` vs `title` |
| 타입 불일치 | `rating: "9"` (문자열) vs `rating: 9` (정수) |
| 필드 누락 | `genre` 필드가 빠짐 |

이런 문제를 **근본적으로** 해결하는 것이 Structured Output입니다.

### 언제 Structured Output을 사용해야 하는가?

| 상황 | 권장 방식 |
|------|----------|
| 응답을 코드에서 파싱해야 할 때 | Structured Output |
| 분류, 라벨링, 점수 매기기 | Structured Output + Literal |
| 데이터 추출 (이름, 날짜, 금액) | Structured Output |
| 자유로운 대화, 설명, 요약 | 일반 텍스트 출력 |
| 창작, 번역 | 일반 텍스트 출력 |

**원칙**: 출력이 사람이 아닌 **코드**가 소비한다면 Structured Output을 사용합니다.

## 1.2 google-genai: response_mime_type

가장 간단한 방법은 `response_mime_type="application/json"`을 설정하는 것입니다.
모델이 반드시 유효한 JSON만 출력하도록 강제합니다.

```python
response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"response_mime_type": "application/json"},
)
data = json.loads(response.text)  # 항상 성공
```

In [ ]:
# response_mime_type으로 JSON 강제
prompt = """다음 영화에 대한 정보를 알려줘.
영화: 인셉션
필드: title, director, year, genre, rating(10점 만점)"""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "thinking_config": {"thinking_budget": 0},
    },
)

print("=== 모델 응답 (순수 JSON) ===")
print(response.text)

# JSON 파싱 — (항상) 성공
data = json.loads(response.text)
print(f"\n=== 파싱 결과 ===")
print(f"type: {type(data).__name__}")
for key, value in data.items():
    print(f"  {key}: {value} ({type(value).__name__})")

=== 모델 응답 (순수 JSON) ===
{"title": "Inception", "director": "Christopher Nolan", "year": 2010, "genre": ["Sci-Fi", "Action", "Thriller"], "rating": 8.8}

=== 파싱 결과 ===
type: dict
  title: Inception (str)
  director: Christopher Nolan (str)
  year: 2010 (int)
  genre: ['Sci-Fi', 'Action', 'Thriller'] (list)
  rating: 8.8 (float)


In [ ]:
# response_mime_type만으로는 스키마를 보장하지 못하는 예시
# 동일한 프롬프트로 여러 번 호출하면 키 이름이 달라질 수 있음
prompts = [
    "대한민국의 수도와 인구를 알려줘",
    "일본의 수도와 인구를 알려줘",
]

for p in prompts:
    resp = client.models.generate_content(
        model=MODEL,
        contents=p,
        config={
            "response_mime_type": "application/json",
            "thinking_config": {"thinking_budget": 0},
        },
    )
    data = json.loads(resp.text)
    print(f"키: {list(data.keys())} ← 호출마다 키 이름이 다를 수 있음")
    print(f"원문: {data}")

키: ['capital', 'population'] ← 호출마다 키 이름이 다를 수 있음
원문: {'capital': '서울', 'population': '9,732,000'}
키: ['capital', 'population'] ← 호출마다 키 이름이 다를 수 있음
원문: {'capital': '도쿄', 'population': '13,960,236 (2023년 기준)'}


> `response_mime_type`만으로는 키 이름, 타입, 필수 필드를 보장할 수 없습니다.
> 모델이 유효한 JSON을 반환하지만, 구조(스키마)는 여전히 프롬프트에 의존합니다.
> 더 확실한 제어를 위해 `response_schema`를 함께 사용합니다.

In [ ]:
# response_schema에 배열(리스트) 타입 사용
# 여러 항목을 한 번에 구조화하여 받기
list_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "capital": {"type": "string"},
            "population_millions": {"type": "number"},
        },
        "required": ["name", "capital", "population_millions"],
    },
}

response = client.models.generate_content(
    model=MODEL,
    contents="동아시아 국가 5개의 이름, 수도, 인구(백만 단위)를 알려줘",
    config={
        "response_mime_type": "application/json",
        "response_schema": list_schema,
        "thinking_config": {"thinking_budget": 0},
    },
)

countries = json.loads(response.text)
print(f"반환 타입: {type(countries).__name__} (길이: {len(countries)})")
for c in countries:
    print(f"  {c['name']}: {c['capital']} ({c['population_millions']}M)")

from pprint import pprint

pprint(countries)

반환 타입: list (길이: 5)
  South Korea: Seoul (51.7M)
  North Korea: Pyongyang (25.9M)
  Japan: Tokyo (125.7M)
  China: Beijing (1412.0M)
  Mongolia: Ulaanbaatar (3.4M)
[{'capital': 'Seoul', 'name': 'South Korea', 'population_millions': 51.7},
 {'capital': 'Pyongyang', 'name': 'North Korea', 'population_millions': 25.9},
 {'capital': 'Tokyo', 'name': 'Japan', 'population_millions': 125.7},
 {'capital': 'Beijing', 'name': 'China', 'population_millions': 1412.0},
 {'capital': 'Ulaanbaatar', 'name': 'Mongolia', 'population_millions': 3.4}]


## 1.3 google-genai: response_schema

`response_schema`를 설정하면 모델이 **정확히 해당 스키마에 맞는** JSON만 생성합니다.
이를 **제어 생성(Controlled Generation)**이라 합니다.

스키마는 두 가지 방식으로 정의할 수 있습니다:
1. JSON Schema (dict)
2. Pydantic 모델 (클래스)

아래 코드는 JSON Schema 방식을 보여줍니다.

In [ ]:
# response_schema — JSON Schema 방식
movie_schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "director": {"type": "string"},
        "year": {"type": "integer"},
        "genre": {"type": "string"},
        "rating": {"type": "number"},
    },
    "required": ["title", "director", "year", "genre", "rating"],
}

response = client.models.generate_content(
    model=MODEL,
    contents="인셉션 영화 정보를 알려줘",
    config={
        "response_mime_type": "application/json",
        "response_schema": movie_schema,
        "thinking_config": {"thinking_budget": 0},
    },
)

data = json.loads(response.text)
print(f"title: {data['title']}")
print(f"director: {data['director']}")
print(f"year: {data['year']} (type: {type(data['year']).__name__})")
print(f"genre: {data['genre']}")
print(f"rating: {data['rating']} (type: {type(data['rating']).__name__})")

title: 인셉션
director: 크리스토퍼 놀란
year: 2010 (type: int)
genre: SF
rating: 8.8 (type: float)


## 1.4 google-genai: Pydantic 모델로 스키마 정의

JSON Schema를 직접 작성하는 대신 **Pydantic 모델**을 사용하면 더 간결하고 타입 안전합니다.
`response_schema`에 Pydantic 클래스를 직접 전달할 수 있습니다.

```python
from pydantic import BaseModel

class Movie(BaseModel):
    title: str
    director: str
    year: int
    genre: str
    rating: float

response = client.models.generate_content(
    ...,
    config={"response_mime_type": "application/json", "response_schema": Movie},
)
```

> **Field description 작성 팁**
> - 값의 범위를 명시하세요: `"평점 (0.0~10.0)"` → 모델이 범위 내 값을 생성
> - 단위를 명시하세요: `"인구 수 (만 명 단위)"` → 숫자의 스케일이 일관됨
> - 언어를 지정하세요: `"한국어로 작성된 제목"` → 다국어 혼동 방지
> - description이 없으면 모델은 필드 이름만으로 추측합니다. `year`는 비교적 명확하지만, `rating`은 5점 만점인지 10점 만점인지 알 수 없습니다

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str
    director: str
    year: int
    genre: str
    rating: float

response = client.models.generate_content(
    model=MODEL,
    contents="기생충 영화 정보를 알려줘",
    config={
        "response_mime_type": "application/json",
        "response_schema": Movie,
        "thinking_config": {"thinking_budget": 0},
    },
)

# JSON → Pydantic 인스턴스로 변환
movie = Movie.model_validate_json(response.text)
print(f"제목: {movie.title}")
print(f"감독: {movie.director}")
print(f"연도: {movie.year}")
print(f"장르: {movie.genre}")
print(f"평점: {movie.rating}")
print(f"\ntype: {type(movie).__name__}")

제목: 기생충
감독: 봉준호
연도: 2019
장르: 드라마
평점: 8.6

type: Movie


### Field의 description

Pydantic의 `Field(description=...)`을 사용하면 모델에게 각 필드의 의미를 알려줄 수 있습니다.
description은 모델이 **어떤 값을 채워야 하는지** 판단하는 데 도움을 줍니다.

In [ ]:
# Field description으로 모델 가이드
class MovieDetailed(BaseModel):
    title: str = Field(description="영화의 한국어 제목")
    original_title: str = Field(description="영화의 원어 제목")
    director: str = Field(description="감독 이름")
    year: int = Field(description="개봉 연도 (4자리 숫자)")
    genre: str = Field(description="주요 장르 1개")
    rating: float = Field(description="IMDb 기준 평점 (0.0~10.0)")
    summary: str = Field(description="영화 줄거리를 2문장 이내로 요약")

response = client.models.generate_content(
    model=MODEL,
    contents="올드보이 영화 정보를 알려줘",
    config={
        "response_mime_type": "application/json",
        "response_schema": MovieDetailed,
        "thinking_config": {"thinking_budget": 0},
    },
)

movie = MovieDetailed.model_validate_json(response.text)
for field_name, value in movie.model_dump().items():
    print(f"{field_name}: {value}")

title: 올드보이
original_title: Oldboy
director: 박찬욱
year: 2003
genre: 네오 느와르
rating: 8.4
summary: 이우진에게 납치 감금되어 15년간 사설 감옥에 갇혔던 오대수는 잃어버린 지난 세월에 대한 분노와 복수심으로 자신을 가둔 사람을 찾아 나선다. 감금 15년 만에 풀려난 오대수는 자신을 가둔 자를 찾아 복수하기 위해 나서고, 그 과정에서 미스터리한 음모와 마주하게 된다.


In [ ]:
# json_schema와 function_calling의 내부 동작 차이 확인
# json_schema: kwargs에 response_format이 포함됨
# function_calling: kwargs에 tools가 포함됨
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
)

print("=== json_schema 방식 내부 ===")
structured_json_check = llm.with_structured_output(Movie, method="json_schema")
print(f"  type: {type(structured_json_check).__name__}")

print("\n=== function_calling 방식 내부 ===")
structured_fc_check = llm.with_structured_output(Movie, method="function_calling")
print(f"  type: {type(structured_fc_check).__name__}")

# 두 방식 모두 같은 Pydantic 인스턴스를 반환하지만,
# 내부 메커니즘이 다릅니다.
print("\n두 방식 모두 같은 타입의 결과를 반환하지만,")
print("json_schema는 Gemini의 제어 생성을, function_calling은 도구 호출을 활용합니다.")

=== json_schema 방식 내부 ===
  type: RunnableSequence

=== function_calling 방식 내부 ===
  type: RunnableSequence

두 방식 모두 같은 타입의 결과를 반환하지만,
json_schema는 Gemini의 제어 생성을, function_calling은 도구 호출을 활용합니다.


## 1.5 LangChain: with_structured_output()

LangChain은 `with_structured_output()` 메서드를 제공합니다.
Pydantic 모델을 전달하면, 반환값이 자동으로 Pydantic 인스턴스가 됩니다.

```python
structured_llm = llm.with_structured_output(PydanticModel)
result = structured_llm.invoke("프롬프트")
# result는 PydanticModel 인스턴스
```

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
)

# with_structured_output 사용
structured_llm = llm.with_structured_output(Movie)

result = structured_llm.invoke("매트릭스 영화 정보를 알려줘")

print(f"type: {type(result).__name__}")
print(f"title: {result.title}")
print(f"director: {result.director}")
print(f"year: {result.year}")
print(f"rating: {result.rating}")

type: Movie
title: 매트릭스
director: 워쇼스키 자매
year: 1999
rating: 8.7


### method 비교: json_schema vs function_calling

`with_structured_output`는 두 가지 내부 메커니즘을 지원합니다:

| method | 내부 동작 | 특징 |
|--------|---------|------|
| `"json_schema"` | Gemini의 response_schema 활용 | 더 안정적, Gemini 전용 |
| `"function_calling"` | Tool Calling 메커니즘 활용 | 범용적, 모든 LLM 지원 |

기본값은 모델에 따라 다릅니다. Gemini에서는 `"json_schema"`가 기본입니다.

In [ ]:
# method 비교
prompt = "인터스텔라 영화 정보를 알려줘"

# json_schema method (Gemini 제어 생성)
structured_json = llm.with_structured_output(Movie, method="json_schema")
result_json = structured_json.invoke(prompt)

# function_calling method (도구 호출)
structured_fc = llm.with_structured_output(Movie, method="function_calling")
result_fc = structured_fc.invoke(prompt)

print("=== json_schema ===")
print(f"  {result_json.title} ({result_json.year}) - {result_json.rating}점")

print("\n=== function_calling ===")
print(f"  {result_fc.title} ({result_fc.year}) - {result_fc.rating}점")

=== json_schema ===
  Interstellar (2014) - 8.7점

=== function_calling ===
  인터스텔라 (2014) - 9.3점


In [ ]:
# Optional vs 기본값: 모델 동작의 차이
from typing import Optional

class WithOptional(BaseModel):
    name: str
    nickname: Optional[str] = Field(default=None, description="별명 (없으면 null)")

class WithDefault(BaseModel):
    name: str
    nickname: str = Field(default="없음", description="별명")

opt_llm = llm.with_structured_output(WithOptional)
def_llm = llm.with_structured_output(WithDefault)

prompt = "이순신 장군에 대해 알려줘"

result_opt = opt_llm.invoke(prompt)
result_def = def_llm.invoke(prompt)

print(f"Optional: nickname = {result_opt.nickname!r} (type: {type(result_opt.nickname).__name__}), raw: {result_opt}")
print(f"Default:  nickname = {result_def.nickname!r} (type: {type(result_def.nickname).__name__}), raw: {result_def}")
print()
print("Optional은 모델이 '모르면' null을 반환합니다.")
print("기본값은 항상 문자열을 강제하므로, 모델이 억지로 값을 만들 수 있습니다.")

Optional: nickname = None (type: NoneType), raw: name='이순신' nickname=None
Default:  nickname = '충무공' (type: str), raw: name='이순신' nickname='충무공'

Optional은 모델이 '모르면' null을 반환합니다.
기본값은 항상 문자열을 강제하므로, 모델이 억지로 값을 만들 수 있습니다.


> 어떤 method를 선택해야 할까?
> - Gemini 전용 서비스라면 `"json_schema"` 권장 (더 안정적)
> - 여러 LLM 제공자를 지원해야 한다면 `"function_calling"` (범용적)
> - 대부분의 경우 기본값을 그대로 사용하면 됩니다

## 1.6 Pydantic 모델 설계 심화

실전에서는 단순한 필드 외에 다양한 타입이 필요합니다.

### Literal (선택지 제한)

`Literal`을 사용하면 필드 값을 특정 선택지로 제한할 수 있습니다.

### 자주 하는 실수

| 실수 | 증상 | 해결 |
|------|------|------|
| `Field(description=...)` 누락 | 필드 값이 부정확하거나 일관성 없음 | 모든 필드에 description 추가 |
| 중첩 3단계 이상 | 모델이 깊은 필드를 빈 값으로 채움 | 2단계까지만 중첩, 나머지는 평면화 |
| `list[str]` 길이 미지정 | 모델이 0개 또는 수십 개를 반환 | description에 "최대 N개" 명시 |
| `float` 범위 미지정 | 0.8인지 80인지 스케일 혼동 | description에 범위 명시 |
| `Optional` 남용 | 모델이 가능한 필드도 null로 반환 | 반드시 필요한 필드는 required로 유지 |

In [ ]:
from typing import Literal, Optional

# Literal로 선택지 제한
class Sentiment(BaseModel):
    text: str = Field(description="분석 대상 텍스트")
    sentiment: Literal["Very Good", "Good","Bad", "Too Bad", "Neutral"] = Field(
        description="감성 분류 결과"
    )
    confidence: float = Field(description="확신도 (0.0~1.0) 긍정적일 수록 1.0 부정적일 수록 0.0에 가깝게 해주세요.")

structured_sentiment = llm.with_structured_output(Sentiment)

# 여러 텍스트 분류
texts = [
    "이 제품 정말 최고입니다! 강력 추천!",
    "배송이 너무 늦어서 화가 납니다.",
    "그냥 보통이에요. 특별할 건 없습니다.",
]

for text in texts:
    result = structured_sentiment.invoke(f"다음 텍스트의 감성을 분석해줘: {text}")
    print(f"[{result.sentiment:>8}] ({result.confidence:.2f}) {result.text[:40]}")

[Very Good] (0.98) 이 제품 정말 최고입니다! 강력 추천!
[ Too Bad] (0.10) 배송이 너무 늦어서 화가 납니다.
[ Neutral] (0.50) 그냥 보통이에요. 특별할 건 없습니다.


### Optional 필드

`Optional`을 사용하면 값이 없을 수 있는 필드를 정의할 수 있습니다.

In [ ]:
# Optional 필드
class PersonInfo(BaseModel):
    name: str = Field(description="사람 이름")
    age: Optional[int] = Field(default=None, description="나이 (언급되지 않으면 null)")
    job: Optional[str] = Field(default=None, description="직업 (언급되지 않으면 null)")
    hobby: Optional[str] = Field(default=None, description="취미 (언급되지 않으면 null)")

structured_person = llm.with_structured_output(PersonInfo)

# 정보가 부분적인 텍스트
result = structured_person.invoke("김민수는 30살 개발자입니다.")
print(f"name: {result.name}")
print(f"age: {result.age}")
print(f"job: {result.job}")
print(f"hobby: {result.hobby}  ← 언급되지 않아 None")

name: 김민수
age: 30
job: 개발자
hobby: None  ← 언급되지 않아 None


> **프로덕션 환경에서의 실패 대응 전략**
> 1. `include_raw=True`로 원본 응답을 항상 로깅합니다
> 2. 파싱 실패 시 최대 2~3회 재시도합니다 (무한 루프 방지)
> 3. 재시도 실패 시 에러 로깅 + 사용자에게 적절한 메시지를 반환합니다
> 4. 재시도 비용이 높으므로, 스키마를 단순화하는 것이 근본적 해결책입니다

In [ ]:
# LCEL 체인에서 Structured Output 사용
from langchain_core.prompts import ChatPromptTemplate

# 프롬프트 템플릿 + 구조화 출력 체인
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화 정보 전문가입니다. 사용자가 묻는 영화의 정보를 정확히 알려주세요."),
    ("human", "{movie_name} 영화 정보를 알려줘"),
])

# LCEL: prompt | structured_llm
chain = prompt_template | llm.with_structured_output(Movie)

# 체인 실행
result = chain.invoke({"movie_name": "쇼생크 탈출"})
print(f"type: {type(result).__name__}")
print(f"title: {result.title}")
print(f"director: {result.director}")
print(f"year: {result.year}")
print(f"rating: {result.rating}")
print()
print("LCEL 체인에서도 with_structured_output이 자연스럽게 동작합니다.")

type: Movie
title: 쇼생크 탈출
director: 프랭크 다라본트
year: 1994
rating: 9.3

LCEL 체인에서도 with_structured_output이 자연스럽게 동작합니다.


### 중첩 모델과 리스트

Pydantic 모델은 다른 모델을 필드로 포함할 수 있습니다.
리스트 필드로 여러 항목을 구조화할 수도 있습니다.

In [ ]:
# 중첩 모델 + 리스트
class Actor(BaseModel):
    name: str = Field(description="배우 이름")
    role: str = Field(description="극중 역할 이름")

class MovieFull(BaseModel):
    title: str = Field(description="영화 제목")
    year: int = Field(description="개봉 연도")
    director: str = Field(description="감독 이름")
    genre: list[str] = Field(description="장르 리스트 (최대 3개)")
    actors: list[Actor] = Field(description="주요 배우 3명")
    rating: float = Field(description="평점 (0.0~10.0)")

structured_full = llm.with_structured_output(MovieFull)
result = structured_full.invoke("기생충 영화의 상세 정보를 알려줘")

print(f"제목: {result.title} ({result.year})")
print(f"감독: {result.director}")
print(f"장르: {', '.join(result.genre)}")
print(f"평점: {result.rating}")
print(f"배우:")
for actor in result.actors:
    print(f"  - {actor.name} ({actor.role})")

제목: 기생충 (2019)
감독: 봉준호
장르: 코미디, 드라마, 스릴러
평점: 8.6
배우:
  - 송강호 (김기택)
  - 최우식 (김기우)
  - 박소담 (김기정)


> Pydantic 모델 설계 팁
> - `Field(description=...)`은 반드시 작성하세요. 모델이 필드의 의미를 이해하는 데 핵심 역할을 합니다
> - `Literal`로 선택지를 제한하면 분류 정확도가 크게 향상됩니다
> - `Optional`은 정보가 불확실할 때 사용하세요. null을 허용하면 모델이 억지로 값을 만들지 않습니다
> - 중첩은 2단계까지가 적절합니다. 너무 깊으면 모델의 출력 정확도가 떨어집니다

> **google-genai와 LangChain 선택 기준 요약**
> - **빠른 프로토타이핑**: google-genai가 더 직접적 (response_schema만 설정)
> - **프로덕션 서비스**: LangChain이 더 편리 (자동 파싱, include_raw, 재시도)
> - **모델 교체 가능성**: LangChain 권장 (인터페이스 통일)
> - **성능 최적화**: google-genai가 약간 유리 (래핑 오버헤드 없음)

## 1.7 include_raw와 실패 핸들링

구조화 출력이 실패할 수 있습니다.
- 모델이 스키마를 완벽히 따르지 못하는 경우
- 네트워크 오류 등으로 불완전한 응답이 도착한 경우

`include_raw=True`를 설정하면 원본 응답과 파싱 결과를 함께 받을 수 있습니다.

In [ ]:
# include_raw=True로 디버깅 정보 확보
structured_raw = llm.with_structured_output(Movie, include_raw=True)
result = structured_raw.invoke("어벤져스 영화 정보를 알려줘")

print(f"result type: {type(result).__name__}")
print(f"keys: {list(result.keys())}")
print()

# 원본 AIMessage
print(f"=== raw (AIMessage) ===")
print(f"  content: {result['raw'].content[:100]}")

# 파싱된 Pydantic 인스턴스
print(f"\n=== parsed (Movie) ===")
print(f"  {result['parsed']}")

# 파싱 에러 (없으면 None)
print(f"\n=== parsing_error ===")
print(f"  {result['parsing_error']}")

result type: dict
keys: ['raw', 'parsed', 'parsing_error']

=== raw (AIMessage) ===
  content: {"title":"어벤져스","director":"조스 웨던","year":2012,"genre":"액션","rating":8.0}

=== parsed (Movie) ===
  title='어벤져스' director='조스 웨던' year=2012 genre='액션' rating=8.0

=== parsing_error ===
  None


### 재시도 패턴

파싱 실패 시 자동으로 재시도하는 패턴입니다.
실패한 경우 원본 응답을 로깅하고, 다시 시도합니다.

```python
result = {
    "parsed": Movie(title="다크나이트", ...),  # 파싱 성공 시 Pydantic 객체
    "raw": AIMessage("..."),                    # LLM의 원본 응답
    "parsing_error": None or Error,             # 파싱 에러 (없으면 None)
}
```

이게 없으면 파싱 실패 시 바로 예외가 터지지만, 있으면 **에러를 잡아서 재시도할 수 있습니다.**

---

## 로직 흐름
```
시도 1 → LLM 호출 → 파싱 성공? → Yes → 결과 반환 ✅
                              → No  → 실패 로그 출력
시도 2 → LLM 호출 → 파싱 성공? → Yes → 결과 반환 ✅
                              → No  → 실패 로그 출력
시도 3 → LLM 호출 → 파싱 성공? → Yes → 결과 반환 ✅
                              → No  → 최종 에러 발생 ❌

```
## 왜 필요한가

LLM은 비결정적(non-deterministic)이라 **같은 프롬프트에도 매번 다른 응답**을 합니다. 예를 들어:
```
시도 1: {"title": "다크나이트", "year": "2008년"}  ← year가 문자열 → 파싱 실패
시도 2: {"title": "다크나이트", "year": 2008}      ← 정상 → 파싱 성공 ✅


In [ ]:
# 재시도 패턴
def invoke_with_retry(structured_llm, prompt, max_retries=3):
    """파싱 실패 시 재시도합니다."""
    structured_raw = structured_llm.with_structured_output(
        Movie, include_raw=True
    ) if not hasattr(structured_llm, '_include_raw') else structured_llm

    for attempt in range(max_retries):
        result = llm.with_structured_output(Movie, include_raw=True).invoke(prompt)

        if result["parsing_error"] is None:
            return result["parsed"]

        print(f"시도 {attempt + 1} 실패: {result['parsing_error']}")
        print(f"  원본: {result['raw'].content[:100]}")

    raise ValueError(f"{max_retries}회 재시도 후에도 파싱 실패")

# 테스트 (정상적인 경우 1회 만에 성공)
movie = invoke_with_retry(llm, "다크나이트 영화 정보를 알려줘")
print(f"성공: {movie.title} ({movie.year})")

성공: The Dark Knight (2008)


## 1.8 스트리밍 Structured Output

구조화 출력도 스트리밍할 수 있습니다.
스트리밍 시 각 청크는 **부분적인 Pydantic 인스턴스** 또는 **dict 조각**입니다.

LangChain의 `with_structured_output`에서 스트리밍하면,
각 청크가 점진적으로 채워지는 인스턴스를 반환합니다.

참고: 모든 모델에서 되는 것이 아니며 gemini-3부터 된다고 공식문서에 나와 있습니다.

https://ai.google.dev/gemini-api/docs/structured-output?hl=ko&example=recipe

In [ ]:
# 스트리밍 structured output
structured_stream = llm.with_structured_output(Movie, method="json_schema")

print("스트리밍 청크:")
for i, chunk in enumerate(structured_stream.stream("반지의 제왕 영화 정보를 제목, 감독, 배우, 요약, 개봉년도 등을 포함해서 알려줘")):
    print(f"  [{i}] type={type(chunk).__name__}, value={chunk}")
    if i >= 6:  # 처음 7개만 표시
        print("  ...")
        break

스트리밍 청크:
  [0] type=Movie, value=title='반지의 제왕' director='피터 잭슨' year=2001 genre='판타지' rating=8.9


In [ ]:
# 스트리밍에서 최종 결과 얻기
final = None
for chunk in structured_stream.stream("반지의 제왕 영화 정보를 알려줘"):
    final = chunk

print(f"최종 결과: {final}")
print(f"type: {type(final).__name__}")

최종 결과: title='The Lord of the Rings: The Fellowship of the Ring' director='Peter Jackson' year=2001 genre='Fantasy' rating=8.8
type: Movie


In [ ]:
class Feedback(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]
    summary: str

client = genai.Client(api_key=GEMINI_API_KEY)
prompt = "The new UI is incredibly intuitive and visually appealing. Great job. Add a very long summary to test streaming!"

response_stream = client.models.generate_content_stream(
    model="gemini-3-flash-preview",
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": Feedback.model_json_schema(),
    },
)

for chunk in response_stream:
    print(chunk.candidates[0].content.parts[0].text)
    print()

{"sentiment":"positive","

summary":"The user provided highly enthusiastic feedback regarding the recent updates to the user interface, noting that the design is both incredibly

 intuitive for navigation and visually appealing to the eye. This positive reception underscores the success of the design team's efforts in

 creating a user-centric experience that balances functionality with aesthetic elegance. Furthermore, the user explicitly praised the work done, reinforcing

 the positive sentiment. In addition to the praise, the user requested an intentionally long summary to test the streaming capabilities of the

 response system. This requirement serves as a stress test for the output generation, ensuring that the system can handle larger payloads of

 text without losing data integrity or performance. By providing such a detailed summary, we confirm that the sentiment analysis engine correctly identifies

 the positive tone of the input while also demonstrating the system's ability

> 스트리밍 Structured Output 주의사항
> - 중간 청크는 불완전한 상태일 수 있습니다 (필드가 None이거나 부분 문자열)
> - 최종 결과는 마지막 청크에서 얻어야 합니다
> - 실시간 UI에 부분 결과를 표시할 때는 None 체크가 필요합니다

## 1.9 google-genai vs LangChain 비교

| 항목 | google-genai | LangChain |
|------|-------------|----------|
| 스키마 정의 | `response_schema` (JSON Schema 또는 Pydantic) | `with_structured_output(Pydantic)` |
| 반환 타입 | JSON 문자열 → 수동 파싱 필요 | 자동으로 Pydantic 인스턴스 |
| 실패 핸들링 | 직접 try/except | `include_raw=True` |
| 스트리밍 | JSON 청크 직접 처리 | 부분 인스턴스 자동 생성 |
| 모델 교체 | Gemini 전용 | 모든 LLM 호환 |

아래 코드는 같은 작업을 두 방식으로 수행합니다.

In [ ]:
# 같은 작업: 두 방식 비교
prompt = "해리포터 영화 정보를 알려줘"

# google-genai 방식
resp_genai = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": Movie,
        "thinking_config": {"thinking_budget": 0},
    },
)
movie_genai = Movie.model_validate_json(resp_genai.text)  # 수동 파싱

# LangChain 방식
movie_lc = structured_llm.invoke(prompt)  # 자동 파싱

print("=== google-genai ===")
print(f"  {movie_genai.title} ({movie_genai.year}) - {movie_genai.rating}점")
print(f"  파싱: 수동 (model_validate_json)")

print("\n=== LangChain ===")
print(f"  {movie_lc.title} ({movie_lc.year}) - {movie_lc.rating}점")
print(f"  파싱: 자동 (with_structured_output)")

=== google-genai ===
  Harry Potter and the Sorcerer's Stone (2001) - 7.6점
  파싱: 수동 (model_validate_json)

=== LangChain ===
  Harry Potter and the Sorcerer's Stone (2001) - 7.6점
  파싱: 자동 (with_structured_output)


---

### 생각해보기

1. 프롬프트만으로 JSON을 요청하는 방식과 response_schema를 사용하는 방식은 비용이 다를까요? 어떤 방식이 더 효율적일까요?
2. `Literal["positive", "negative", "neutral"]`과 `str`의 차이는 무엇일까요? 분류 작업에서 Literal을 사용하면 어떤 이점이 있을까요?
3. 중첩 모델이 3단계 이상 깊어지면 모델의 출력 정확도가 떨어질 수 있습니다. 이를 어떻게 해결할 수 있을까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 8-1: google-genai response_schema로 JSON 강제

google-genai의 response_schema를 사용하여 구조화된 JSON 응답을 받으세요.

**요구사항**
1. 프롬프트: "파이썬 프로그래밍 언어에 대해 알려줘"
2. 스키마: name(str), year_created(int), creator(str), main_use(str)
3. `response_mime_type`과 `response_schema` 설정
4. JSON 파싱 후 각 필드를 출력

In [ ]:
# TODO: google-genai response_schema 사용
prompt = "파이썬 프로그래밍 언어에 대해 알려줘"

# 정답
lang_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "year_created": {"type": "integer"},
        "creator": {"type": "string"},
        "main_use": {"type": "string"},
    },
    "required": ["name", "year_created", "creator", "main_use"],
}

response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": lang_schema,
        "thinking_config": {"thinking_budget": 0},
    },
)

data = json.loads(response.text)
for key, value in data.items():
    print(f"{key}: {value}")

print("TODO를 완성해주세요")

### 실습 8-2: Pydantic 모델로 스키마 정의

Pydantic 모델을 정의하고 google-genai의 response_schema에 전달하세요.

**요구사항**
1. `BookInfo` 모델 정의: title(str), author(str), year(int), pages(int), summary(str)
2. 각 필드에 `Field(description=...)` 추가
3. 프롬프트: "해리포터와 마법사의 돌 책 정보를 알려줘"
4. `model_validate_json()`으로 파싱 후 출력

In [ ]:
from pydantic import BaseModel, Field

# TODO: Pydantic 모델 정의 + response_schema 사용

# 정답
class BookInfo(BaseModel):
    title: str = Field(description="책 제목")
    author: str = Field(description="저자 이름")
    year: int = Field(description="출판 연도")
    pages: int = Field(description="페이지 수")
    summary: str = Field(description="줄거리를 2문장 이내로 요약")

response = client.models.generate_content(
    model=MODEL,
    contents="해리포터와 마법사의 돌 책 정보를 알려줘",
    config={
        "response_mime_type": "application/json",
        "response_schema": BookInfo,
        "thinking_config": {"thinking_budget": 0},
    },
)

book = BookInfo.model_validate_json(response.text)
for field, value in book.model_dump().items():
    print(f"{field}: {value}")

print("TODO를 완성해주세요")

### 실습 8-3: LangChain with_structured_output

LangChain의 with_structured_output()을 사용하여 구조화된 출력을 받으세요.

**요구사항**
1. `CountryInfo` 모델 정의: name(str), capital(str), population(int), language(str), continent(str)
2. `llm.with_structured_output(CountryInfo)` 사용
3. 프롬프트: "대한민국에 대해 알려줘"
4. 반환된 Pydantic 인스턴스의 각 필드를 출력

In [ ]:
# TODO: LangChain with_structured_output

# 정답
class CountryInfo(BaseModel):
    name: str = Field(description="국가 이름")
    capital: str = Field(description="수도")
    population: int = Field(description="인구 수 (대략적)")
    language: str = Field(description="공용어")
    continent: str = Field(description="대륙")

structured_country = llm.with_structured_output(CountryInfo)
result = structured_country.invoke("대한민국에 대해 알려줘")

print(f"type: {type(result).__name__}")
for field, value in result.model_dump().items():
    print(f"{field}: {value}")

print("TODO를 완성해주세요")

### 실습 8-4: Literal로 분류 작업

Literal을 활용하여 텍스트를 정해진 카테고리로 분류하세요.

**요구사항**
1. `Classification` 모델: text(str), category(Literal[...]), confidence(float)
2. category 선택지: "technology", "sports", "entertainment", "science", "politics"
3. 3개의 서로 다른 뉴스 헤드라인을 분류
4. 결과를 표 형태로 출력

In [ ]:
from typing import Literal

# TODO: Literal 분류 작업

# 정답
class Classification(BaseModel):
    text: str = Field(description="원본 텍스트")
    category: Literal["technology", "sports", "entertainment", "science", "politics"] = Field(
        description="분류 카테고리"
    )
    confidence: float = Field(description="확신도 (0.0~1.0)")

structured_cls = llm.with_structured_output(Classification)

headlines = [
    "AI 반도체 기업, 사상 최대 매출 달성",
    "손흥민, 프리미어리그 100호 골 달성",
    "화성 탐사 로버, 물 흔적 발견",
]

for headline in headlines:
    result = structured_cls.invoke(f"다음 뉴스를 분류해줘: {headline}")
    print(f"[{result.category:>15}] ({result.confidence:.2f}) {headline}")

print("TODO를 완성해주세요")

### 실습 8-5: 중첩 Pydantic 모델

중첩된 Pydantic 모델을 설계하여 복잡한 구조화 출력을 받으세요.

**요구사항**
1. `Ingredient` 모델: name(str), amount(str)
2. `Recipe` 모델: dish_name(str), cuisine(str), difficulty(Literal["easy","medium","hard"]), ingredients(list[Ingredient]), steps(list[str])
3. 프롬프트: "김치볶음밥 레시피를 알려줘"
4. 결과를 보기 좋게 출력

In [ ]:
# TODO: 중첩 Pydantic 모델

# 정답
class Ingredient(BaseModel):
    name: str = Field(description="재료 이름")
    amount: str = Field(description="필요한 양 (예: 1컵, 2큰술)")

class Recipe(BaseModel):
    dish_name: str = Field(description="요리 이름")
    cuisine: str = Field(description="요리 종류 (한식, 양식 등)")
    difficulty: Literal["easy", "medium", "hard"] = Field(description="난이도")
    ingredients: list[Ingredient] = Field(description="필요한 재료 목록")
    steps: list[str] = Field(description="조리 단계 (순서대로)")

structured_recipe = llm.with_structured_output(Recipe)
recipe = structured_recipe.invoke("김치볶음밥 레시피를 알려줘")

print(f"{recipe.dish_name} ({recipe.cuisine}) - 난이도: {recipe.difficulty}")
print(f"\n재료:")
for ing in recipe.ingredients:
    print(f"  - {ing.name}: {ing.amount}")
print(f"\n조리 단계:")
for i, step in enumerate(recipe.steps, 1):
    print(f"  {i}. {step}")

print("TODO를 완성해주세요")

### 실습 8-6: include_raw로 디버깅

`include_raw=True`를 사용하여 원본 응답과 파싱 결과를 함께 확인하세요.

**요구사항**
1. `Movie` 모델과 `include_raw=True` 사용
2. 프롬프트: "타이타닉 영화 정보를 알려줘"
3. result의 세 가지 키(raw, parsed, parsing_error)를 각각 출력
4. parsing_error가 None인지 확인

In [ ]:
# TODO: include_raw로 디버깅

# 정답
structured_debug = llm.with_structured_output(Movie, include_raw=True)
result = structured_debug.invoke("타이타닉 영화 정보를 알려줘")

print(f"=== raw ===")
print(f"  type: {type(result['raw']).__name__}")
print(f"  content: {result['raw'].content[:100]}")

print(f"\n=== parsed ===")
print(f"  type: {type(result['parsed']).__name__}")
print(f"  {result['parsed']}")

print(f"\n=== parsing_error ===")
print(f"  {result['parsing_error']}")
print(f"  파싱 성공: {result['parsing_error'] is None}")

print("TODO를 완성해주세요")

### 실습 8-7: 스트리밍 Structured Output

구조화 출력을 스트리밍으로 받아보세요.

**요구사항**
1. `Movie` 모델로 `with_structured_output` 구성
2. `.stream()`으로 스트리밍 실행
3. 각 청크의 타입과 값을 출력 (처음 5개)
4. 마지막 청크에서 최종 결과를 확인

In [ ]:
# TODO: 스트리밍 Structured Output

# 정답
structured_s = llm.with_structured_output(Movie)
final = None

for i, chunk in enumerate(structured_s.stream("아바타 영화 정보를 알려줘")):
    final = chunk
    if i < 5:
        print(f"[{i}] type={type(chunk).__name__}, value={chunk}")

print(f"\n최종 결과:")
print(f"  type: {type(final).__name__}")
if hasattr(final, 'title'):
    print(f"  title: {final.title}")
    print(f"  year: {final.year}")
    print(f"  rating: {final.rating}")

print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 8-4에서 Literal 대신 str을 사용하면 어떤 문제가 생길 수 있을까요? "tech"와 "technology"처럼 비표준 값이 나올 수 있을까요?
2. 실습 8-5의 레시피 모델에서 steps를 `list[str]` 대신 별도의 Step 모델(`step_number: int, description: str`)로 정의하면 어떤 장단점이 있을까요?
3. include_raw를 항상 True로 설정하면 어떤 단점이 있을까요? 프로덕션 환경에서는 어떻게 사용해야 할까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 8-1: 복잡한 중첩 스키마 설계 + 파싱 실패 대응 (난이도: 상)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

"회사 채용 공고" 정보를 구조화하는 복잡한 Pydantic 모델을 설계하세요.

**스키마 요구사항**
- 회사 정보: company_name, industry, location
- 포지션: title, department, level(Literal["junior","mid","senior","lead"])
- 요구사항: required_skills(list[str]), preferred_skills(list[str]), min_experience_years(int)
- 혜택: salary_range(Optional[str]), benefits(list[str])

**과제**
- Pydantic 모델 설계 (중첩 구조 활용)
- 3개의 서로 다른 채용 공고 텍스트를 구조화
- include_raw로 파싱 성공 여부 확인
- 파싱 실패 시 재시도 로직 구현

**힌트**
- 회사 정보, 포지션, 요구사항, 혜택을 각각 별도의 Pydantic 모델로 분리하세요
- Field(description=...)을 상세하게 작성하면 모델의 정확도가 높아집니다

In [ ]:
# 여기에 코드를 작성하세요
# === 챌린지 8-1 답안 ===
from pydantic import BaseModel, Field
from typing import Literal, Optional

# ── 1단계: Pydantic 모델 설계 (중첩 구조) ──
class Company(BaseModel):
    company_name: str = Field(description="회사 이름")
    industry: str = Field(description="업종 (예: IT, 금융, 제조)")
    location: str = Field(description="근무지")

class Position(BaseModel):
    title: str = Field(description="포지션명")
    department: str = Field(description="소속 부서")
    level: Literal["junior", "mid", "senior", "lead"] = Field(description="직급 레벨")

class Requirements(BaseModel):
    required_skills: list[str] = Field(description="필수 기술 스택 (최대 5개)")
    preferred_skills: list[str] = Field(description="우대 사항 (최대 3개)")
    min_experience_years: int = Field(description="최소 경력 연수")

class Benefits(BaseModel):
    salary_range: Optional[str] = Field(default=None, description="연봉 범위 (공개되지 않으면 null)")
    benefits: list[str] = Field(description="복리후생 목록 (최대 5개)")

class JobPosting(BaseModel):
    company: Company
    position: Position
    requirements: Requirements
    benefits: Benefits


# ── 2단계: 재시도 로직 ──
def parse_job_posting(text, max_retries=3):
    for attempt in range(max_retries):
        result = llm.with_structured_output(JobPosting, include_raw=True).invoke(
            f"다음 채용 공고를 분석해주세요:\n\n{text}"
        )
        if result["parsing_error"] is None:
            return result["parsed"]
        print(f"  시도 {attempt+1} 실패: {result['parsing_error']}")
    raise ValueError("파싱 실패")


# ── 3단계: 3개 채용 공고 구조화 ──
postings = [
    "네이버에서 백엔드 개발자를 채용합니다. IT업종, 판교 근무. 시니어급. 검색플랫폼팀 소속. Python, Django, Redis 필수. Kubernetes 우대. 경력 5년 이상. 연봉 7000~9000만원. 자율출퇴근, 식대지원, 건강검진 제공.",
    "카카오뱅크 데이터 엔지니어 채용. 금융IT, 판교. 미드레벨, 데이터플랫폼팀. Spark, Airflow, SQL 필수. AWS 우대. 3년 이상. 연봉 비공개. 스톡옵션, 점심제공, 교육비지원.",
    "토스 주니어 프론트엔드 개발자 모집. 핀테크, 서울 강남. 프로덕트팀. React, TypeScript 필수. Next.js 우대. 1년 이상. 연봉 4500~6000만원. 자율출퇴근, 장비지원, 건강검진.",
]

for i, text in enumerate(postings, 1):
    job = parse_job_posting(text)
    print(f"[공고 {i}] {job.company.company_name} - {job.position.title} ({job.position.level})")
    print(f"  필수: {', '.join(job.requirements.required_skills)}")
    print(f"  연봉: {job.benefits.salary_range or '비공개'}")
    print()

---

### 생각해보기

1. 채용 공고마다 형식이 다른데(어떤 공고는 연봉을 공개하고, 어떤 공고는 안 하고), 하나의 스키마로 모든 공고를 파싱할 수 있었나요? Optional 필드가 이 문제를 어떻게 해결하나요?
2. Structured Output을 사용하면 모델의 창의성이 제한될 수 있습니다. 구조화가 필요한 작업과 자유 텍스트가 더 적합한 작업의 경계는 어디일까요?
3. 실제 서비스에서 Structured Output 파싱이 실패했을 때, 사용자에게 어떤 경험을 제공해야 할까요? 에러 메시지? 재시도? 자유 텍스트 폴백?